# Séance 8 - Le Bivarié I

Voir user guide :
- Marks (2/3) : https://altair-viz.github.io/user_guide/marks/index.html

## Objectifs
- Visualiser une relation entre deux variables, ou plus
- Créer des **groupes** avec **pd.cut** et comparaison avec pd.crosstab
- Comprendre la différence entre l'utilisation de diagrammes en barre ou lignes

Rappel : Nous utilisons **pandas** pour préparer, transformer et résumer les données.

## Pourquoi le bivarié ?

L'analyse bivariée permet d'explorer la **relation entre deux variables**. Cela nous permet de répondre à des questions plus intéressantes comme :
- Le niveau d'éducation influence-t-il le revenu ?
- L'âge est-il lié à la préférence partisane ?
- La polarisation politique varie-t-elle selon les groupes sociaux ?

**Bonnes pratiques** :
- Toujours se demander : Quelle variable expliquer (Y) par quelle variable explicative (X) ?
- Choisir le type de graphique selon la nature des variables (Q/Q, Q/O, O/O)
- Ajouter des éléments de contexte (titres, exemple de lecture, source)

In [ ]:
# ===========================================
# Installation & Chargement des bibliothèques
# ===========================================
%pip install "vegafusion[embed]>=1.5.0" "vl-convert-python>=1.6.0"

import pandas as pd
import altair as alt
import warnings
warnings.filterwarnings('ignore')

# Configuration d'Altair
alt.data_transformers.enable("vegafusion")  # exécution locale
alt.data_transformers.disable_max_rows()  # datasets large

In [46]:
# ===========================================
# Chargement de la base de données 
# ===========================================
data_url = "https://raw.githubusercontent.com/datamisc/ts-2024/main/data.csv"
df_raw = pd.read_csv(data_url, compression="gzip", low_memory=False)

# Sélectionner quelques variables d'intérêt
my_vars = [

    "V241156",  # thermomètre Harris
    "V241157",  # thermomètre Trump
    "V241177",  # idéologie
    "V241043",  # intention de vote
    "V241465x",  # éducation
    "V241566x",  # revenu (19 catégories - recodage nécessaire)
    "V241567x",  # revenu (6 catégories - déjà codé)
    "V241458x",  # âge
    "V241501x",  # ethnie
]

df = df_raw[my_vars].copy()

df.columns = [
    "thermo_harris",
    "thermo_trump",
    "ideology",
    "vote_int",
    "education",
    "income_raw",
    "income_cat",
    "age",
    "ethnicity",
]

df.head()

,thermo_harris,thermo_trump,ideology,vote_int,education,income_raw,income_cat,age,ethnicity
0,0,100,6,2,4,27,5,50,3
1,50,50,4,3,3,26,5,41,4
2,90,0,2,1,3,24,5,44,1
3,50,70,99,2,3,12,3,45,4
4,5,60,4,2,5,15,3,80,1


## Nuage de points (scatterplot)

### Quand l'utiliser ?
- Pour explorer la relation entre **deux variables quantitatives continues**
- Permet de visualiser la forme de la relation (linéaire, exponentielle, etc.)
- Utile pour détecter des valeurs aberrantes ou des clusters
### Bonnes pratiques :
- Utiliser la transparence (`opacity=..`) pour gérer la superposition des points
- Utiliser des couleurs/formes (`shape=..` et/ou `color=..`) pour représenter d'autre dimensions
- Nous pouvons aussi ajouter une droite de régression linéaire pour montrer la tendance (plus tard...)

**Exemple** : Relation entre les thermomètres Harris et Trump

In [ ]:
# Préparation des données
mask = (df['thermo_harris'].between(0, 100)) & (df['thermo_trump'].between(0, 100))
df_scatter = df[mask][['thermo_harris', 'thermo_trump']]

df_scatter

In [ ]:
alt.Chart(df_scatter).mark_point(size=25, opacity=0.35).encode(
    x=alt.X('thermo_trump', type='quantitative', title='Thermomètre Trump', scale=alt.Scale(domain=[0, 100])),
    y=alt.Y('thermo_harris', type='quantitative', title='Thermomètre Harris', scale=alt.Scale(domain=[0, 100]))
).properties(
    title=alt.TitleParams(
        text="Évaluations de Harris et Trump une échelle de sympathie",
        subtitle=[
        # Mini hack-time: comment faire en sorte que le sous-titre aille à la ligne?
            "Les points dans le quadrant supérieur gauche représentent les répondants qui évaluent Harris très positivement et Trump très négativement sur l’échelle de sympathie.",
            "Source : ANES 2024 Time Series Study"
        ],
        anchor='start'
    ),
    width=400,
    height=400
)

**Interpretation** : On observe une relation negative : aimer Trump va souvent avec moins aimer Harris. 

### Hack-Time 

Ajoutez une couleur selon l'intention de vote (`vote_int`) en creant d'abord un nouveau tableau (df2) avec les collonnes suivantes:
- les deux thermomètres (0-100)
- vote_int (> 0)

Puis utilisez le paramètre:
- `color = alt.Color('vote_int', type='nominal')`
- gardez l'echelle 0-100

In [ ]:
# Hack-Time 


## Barplot de moyennes

### Quand l'utiliser ?
- Pour visualiser une relation entre une **variable categorielle (X)** et une **variable quantitative (Y)**
- Pour une relation categorielle -> quantitative
- Préférable quand les categories ne sont pas ordonnées 
- Plus lisible pour un petit nombre de categories

### Line vs Barplot ?
- **Line** : préférer pour des categories ordonnées (idéologie, âge, revenu)
- **Barplot** : préférer pour des categories sans ordre naturel (région, parti, ethnie)

- Si votre objectif principal est de comparer des catégories utilisez un bar plot.
- Si vous voulez mettre l’accent sur la tendance et l’augmentation/diminution graduelle utilisez un line plot.

**Exemple** : Revenu moyen selon l'origine ethnique

La variable `income_raw` (V241566x) contient 28 categories de revenu. 

La variable `ethnicity` (V241501x) contient 6 categories (concentrons nous sur les 3 premières).

In [80]:
# Verifier les valeurs de income_raw
df['income_raw'].value_counts().sort_index()

income_raw
-9     301
-5      10
-1     245
 1     405
 2      51
 3     108
 4      37
 5      75
 6      31
 7     104
 8      45
 9      92
 10     29
 11    171
 12    149
 13    199
 14    124
 15    233
 16    102
 17    204
 18     98
 19    156
 20    142
 21    262
 22    194
 23    331
 24    268
 25    260
 26    297
 27    416
 28    382
Name: count, dtype: int64

Maintenant, calculons le revenu moyen par origine ethnique et visualisons avec un barplot :

In [82]:
# Calcul des moyennes de revenu par education

df_inc = df[
    (df['ethnicity'].between(1,3)) & (df['income_raw'].between(1, 28))
][['ethnicity', 'income_raw']]

df_means = (
    df_inc
    .groupby('ethnicity', as_index=False)['income_raw']
    .mean()
)

df_means['ethnicity'] = df_means['ethnicity'].replace({1:"White", 2:"Black", 3:"Hispanic"})

df_means

,ethnicity,income_raw
0,White,18.550126
1,Black,13.127193
2,Hispanic,16.150476


In [83]:
alt.Chart(df_means).mark_bar(color='#BE8400').encode(
    x=alt.X('ethnicity', type='nominal', title=""),
    y=alt.Y('income_raw', type='quantitative', title='Code de Revenu Moyen')
).properties(
    title=alt.TitleParams(
        text="Revenu moyen du foyer par origine ethnique",
        subtitle=[
            "Le revenu moyen des Hispaniques se situe entre 40 000 $ et 44 000 $.",
            "Source : ANES 2024 Time Series Study"
        ],
        anchor='start'
    ),
    width=300,
    height=300
)

alt.Chart(...)

## Ligne de moyennes (line plot)

### Quand l'utiliser ?
- Également pour une relation categorielle -> quantitative
- Et lorsque nous souhaitons montrer des **tendances** ou des **évolutions**
- Permet de comparer facilement plusieurs groupes
### Bonnes pratiques :
- Bien ordonner les catégories sur l'axe X (ne pas se fier à l'ordre par defaut)
- Ajuster les points pour montrer les valeurs exactes
- Utiliser des couleurs/formes pertinantes

**Exemple** : Évaluation moyenne de Harris selon l'idéologie politique

In [84]:
# Calcul des moyennes avec pandas
mask = (df['ideology'].between(1, 7)) & (df['thermo_harris'].between(0, 100))
vars = ['ideology', 'thermo_harris']
df_mean = df[mask][vars]

means = (
    df_mean
    .groupby('ideology', as_index=False)['thermo_harris']
    .mean()
)

means

,ideology,thermo_harris
0,1,76.516000
1,2,82.844553
2,3,72.939227
3,4,54.752961
4,5,31.141414
5,6,11.281553
6,7,7.750000


In [94]:
alt.Chart(means).mark_line(
    point=True, 
    size=3, 
    color='#1D4ED8'
).encode(
    x=alt.X('ideology', type='ordinal', title='Idéologie (1=libéral, 7=conservateur)'),
    y=alt.Y('thermo_harris', type='quantitative', title='Thermomètre de sympathie', scale=alt.Scale(domain=[0, 100]))
).properties(
    title=alt.TitleParams(
        text="Évolution moyenne de la sympathie en vers Harris selon l'idéologie",
        subtitle=[
            "Les respondants d'idéologie 1-2 donnent en moyenne ~80 au thermomètre Harris.",
            "Source : ANES 2024 Time Series Study"
        ],
        anchor='start'
    ),
    width=520,
    height=300
)

alt.Chart(...)

**Interpretation** : Plus on va vers la droite politique, plus l'évaluation moyenne de Harris diminue. Relation monotone et très marquée.

### Hack-Time 2 (15 min)

Reproduisez le même graphique mais pour Trump (`thermo_trump`).
- calculez la moyenne du thermomètre Trump par idéologie avec pandas
- tracez une ligne (échelle 0-100)
- écrivez un titre + sous-titre avec exemple de lecture ou interprétation

Bonus : Tracez les deux lignes (Harris et Trump) sur le même graphique pour comparer :
- Créez un dataframe avec les deux moyennes
- Utilisez `color` pour distinguer Harris et Trump

In [95]:
# Hack-Time 2 : votre code ici

Pour un public scientifique/académique vous pouvez également utiliser une boîte à moustaches.

In [96]:
df_box = df[['ideology', 'thermo_harris']]
df_box = df_box[df_box['ideology'].between(1,7)]
df_box = df_box[df_box['thermo_harris']>0]

alt.Chart(df_box).mark_boxplot(
).encode(
    x=alt.X('ideology', type='ordinal', title='Idéologie'),
    y=alt.Y('thermo_harris', type='quantitative', title='Thermomètre Harris', scale=alt.Scale(domain=[0, 100]))
).properties(
    title=alt.TitleParams(
        text="Évolution de la sympathie en vers Harris selon l'idéologie",
        subtitle=[
            "Source : ANES 2024 Time Series Study"
        ],
        anchor='start'
    ),
    width=520,
    height=300
)

alt.Chart(...)

## Est-ce utile d'avoir un diplôme?


Comme nous l'avons vu, la variable `income_raw` (V241566x) contient 19 categories de revenu. Nous pouvons la recoder en groupes plus lisibles avec **pd.cut** :

In [114]:
# Création de categories de revenu avec pd.cut
# Les codes 1-19 correspondent a des tranches de revenus
income_labels = [
    "Moins de 20k", 
    "$20k-$-$49.9k", 
    "$50k-$99.9k", 
    "$100k-$149.9k", 
    "$150k-$249.9k", 
    "$250k et plus"
]

# On utilise les bins pour les codes 1-15
df['income_recode'] = pd.cut(
    df['income_raw'],
    bins=[0, 6, 14, 22, 25, 27, 28],
    labels=income_labels
)

# Verification
df['income_recode'].value_counts()


income_recode
$50k-$99.9k      1391
$20k-$-$49.9k     913
$100k-$149.9k     859
$150k-$249.9k     713
Moins de 20k      707
$250k et plus     382
Name: count, dtype: int64

In [126]:
# Filtrage des données 
mask = df['education'] > 0
df_income = df[mask]

vars = ['education', 'income_recode']
df_income = df[vars]
df_income

,education,income_recode
0,4,$150k-$249.9k
1,3,$150k-$249.9k
2,3,$100k-$149.9k
3,3,$20k-$-$49.9k
4,5,$50k-$99.9k
...,...,...
5516,5,Moins de 20k
5517,3,$50k-$99.9k
5518,3,$20k-$-$49.9k
5519,4,$50k-$99.9k


In [127]:
# Calcul des moyennes 
income_means = (
    df_income
    .groupby('income_recode', as_index=False)['education']
    .mean()
)
income_means

,income_recode,education
0,Moins de 20k,2.557284
1,$20k-$-$49.9k,2.879518
2,$50k-$99.9k,3.269590
3,$100k-$149.9k,3.681024
4,$150k-$249.9k,4.021038
5,$250k et plus,4.104712


Notez que dans l'ANES, il existe des déjà des variables regroupées! La variable V241567x est déjà codéee en 6 catégories.

In [ ]:
alt.Chart(inc_means2).mark_line(color='#059669').encode(
    x=alt.X('education', type='ordinal', title="Niveau d'education"),
    y=alt.Y('income_recode', type='quantitative', title='Revenu')
).properties(
    title=alt.TitleParams(
        text="Education et revenu moyen",
        subtitle=[
            "Le revenu moyen augmente avec l'education.",
            "Source : ANES 2024 Time Series Study"
        ],
        anchor='start'
    ),
    width=520,
    height=300
)

alt.Chart(...)

In [108]:
df['income_cat'].value_counts().sort_index()

income_cat
-9     297
-5      10
-4      14
 1     474
 2     564
 3    1032
 4    1114
 5    1620
 6     396
Name: count, dtype: int64

In [109]:
# Labels des categories de revenu (deja codees)
revenu_labels = {
    1: "Moins de $25k",
    2: "$25k-$50k",
    3: "$50k-$100k",
    4: "$100k-$150k",
    5: "$150k-$250k",
    6: "$250k et plus",
}

# Calcul des moyennes avec income_cat
df_inc2 = df[
    (df['education'] > 0) & (df['income_cat'].between(1, 6))
][['education', 'income_cat']]

inc_means2 = (
    df_inc2
    .groupby('education', as_index=False)['income_cat']
    .mean()
)
inc_means2

,education,income_cat
0,1,2.400778
1,2,3.102198
2,3,3.576877
3,4,4.173047
4,5,4.552987


In [111]:
alt.Chart(inc_means2).mark_line(color='#059669').encode(
    x=alt.X('education', type='ordinal', title="Niveau d'education"),
    y=alt.Y('income_cat', type='quantitative', title='Revenu')
).properties(
    title=alt.TitleParams(
        text="Education et revenu moyen",
        subtitle=[
            "Le revenu moyen augmente avec l'education.",
            "Source : ANES 2024 Time Series Study"
        ],
        anchor='start'
    ),
    width=520,
    height=300
)

alt.Chart(...)

**Interpretation** : Il existe une relation positive entre niveau d'education et revenu. Les personnes avec un niveau d'education plus eleve ont en moyenne un revenu plus eleve.

Note : Les deux approches sont valides. pd.cut offre plus de flexibilite, tandis que les variables deja codees sont plus simples a utiliser.

### Hack-Time 3 (15 min)

**Exercice A** : Representez la moyenne du revenu (`income_cat`) par niveau d'education avec un **line plot** au lieu d'un barplot.

**Exercice B** : Comment le revenu evolue-t-il selon le genre ou l'ethnicite ?

Ajoutez : titre + sous-titre avec exemple de lecture + source.

In [ ]:
# Hack-Time : votre code ici


## 4) Polarisation et âge

### Création d'une variable composite

La **polarisation** peut etre mesuree par la difference absolue entre les thermomètres Trump et Harris. Cela capture l'intensite de la preference politique (peu importe le candidat prefere, les personnes polarisees ont une opinion très tranchée).

In [ ]:
# Création de la variable polarisation (valeur ABSOLUE de la difference)
df['polarisation'] = abs(df['thermo_trump'] - df['thermo_harris'])

# Verification
df['polarisation'].describe()

### Utilisation de pd.cut pour les groupes d'age

In [ ]:
# Verification de la variable age
print("Variable age - min/max :", df['age'].min(), "/", df['age'].max())
print("Valeurs manquantes :", df['age'].isna().sum())

In [ ]:
# Création de categories d'age avec pd.cut
df['age_group'] = pd.cut(
    df['age'],
    bins=[17, 29, 44, 59, 74, 100],
    labels=['18-29 ans', '30-44 ans', '45-59 ans', '60-74 ans', '75+ ans']
)

# Calcul de la polarisation moyenne par groupe d'age
df_pol = df[
    df['polarisation'].between(0, 100)
][['age_group', 'polarisation']]

pol_means = (
    df_pol
    .groupby('age_group', as_index=False)['polarisation']
    .mean()
)

pol_means

In [ ]:
alt.Chart(pol_means).mark_line(color='#7C3AED').encode(
    x=alt.X('age_group', type='nominal', title='Groupe d\'age'),
    y=alt.Y('polarisation', type='quantitative', title='Polarisation (moyenne)')
).properties(
    title=alt.TitleParams(
        text="Polarisation politique selon l'age",
        subtitle=[
            "Exemple de lecture : Les 18-29 ans ont une polarisation moyenne de ~47, les 75+ ans de ~56. Plus la valeur est elevee, plus les opinions sont trichees.",
            "Source : ANES 2024 Time Series Study"
        ],
        anchor='start'
    ),
    width=520,
    height=300
)

**Interpretation** : La polarisation augmente avec l'age. Les personnes agees ont des opinions plus trichees (plus proches de 0 ou 100 pour un candidat et l'inverse pour l'autre).

Note : La polarisation est ici mesuree comme |Trump - Harris|, donc 0 = pas de preference, 100 = opinion maximale dans un sens ou l'autre.